In [1]:
import os, pymongo, happybase, pyspark

print("Spark:", pyspark.__version__)

m = pymongo.MongoClient("mongodb://mongo:27017", serverSelectionTimeoutMS=5000)
print("Mongo ping:", m.admin.command("ping"))

h = happybase.Connection("hbase", port=9090)
print("HBase tables:", h.tables())

print("Data files:", sorted(os.listdir("/home/jovyan/work/data")))

Spark: 4.3.0
Mongo ping: {'ok': 1.0}
HBase tables: []
Data files: ['categories.json', 'products.json', 'sessions_0.json', 'sessions_1.json', 'transactions.json', 'users.json']


In [3]:
import json, collections
D = "/home/jovyan/work/data/"

users      = json.load(open(D + "users.json"))
products   = json.load(open(D + "products.json"))
categories = json.load(open(D + "categories.json"))
txns       = json.load(open(D + "transactions.json"))

print("users:", len(users), "| products:", len(products),
      "| categories:", len(categories),
      "| subcategories:", sum(len(c["subcategories"]) for c in categories),
      "| transactions:", len(txns))

linked = sum(1 for t in txns if t["session_id"])
print("transactions linked to a session:", linked, "| unlinked:", len(txns) - linked)
print("transaction status:", dict(collections.Counter(t["status"] for t in txns)))
print("payment methods:", dict(collections.Counter(t["payment_method"] for t in txns)))
print("total revenue:", round(sum(t["total"] for t in txns), 2))
print("avg items per transaction:", round(sum(len(t["items"]) for t in txns) / len(txns), 2))
print("transaction dates:", min(t["timestamp"] for t in txns)[:10], "to", max(t["timestamp"] for t in txns)[:10])
print("registration dates:", min(u["registration_date"] for u in users)[:10], "to", max(u["registration_date"] for u in users)[:10])
print("inactive products:", sum(1 for p in products if not p["is_active"]),
      "| zero stock:", sum(1 for p in products if p["current_stock"] == 0))
print("product fields:", list(products[0].keys()))

users: 10000 | products: 5000 | categories: 25 | subcategories: 109 | transactions: 100000
transactions linked to a session: 40488 | unlinked: 59512
transaction status: {'completed': 55249, 'processing': 15016, 'delivered': 14884, 'shipped': 14851}
payment methods: {'crypto': 10167, 'paypal': 25094, 'gift_card': 14767, 'bank_transfer': 14934, 'credit_card': 24937, 'apple_pay': 10101}
total revenue: 111102382.12
avg items per transaction: 2.27
transaction dates: 2026-08-06 to 2026-11-04
registration dates: 2026-02-07 to 2026-08-06
inactive products: 234 | zero stock: 376
product fields: ['product_id', 'name', 'category_id', 'base_price', 'current_stock', 'is_active', 'price_history', 'creation_date']


In [4]:
%%writefile /home/jovyan/work/load_mongo.py
import json, os, sys
from datetime import datetime
import pymongo

D = sys.argv[1] if len(sys.argv) > 1 else "/home/jovyan/work/data"
URI = sys.argv[2] if len(sys.argv) > 2 else "mongodb://mongo:27017"
db = pymongo.MongoClient(URI)["ecommerce"]

def load(name):
    with open(os.path.join(D, name)) as f:
        return json.load(f)

def dt(s):
    return datetime.fromisoformat(s) if s else None

def insert_batches(coll, docs, size=10000):
    coll.drop()
    for i in range(0, len(docs), size):
        coll.insert_many(docs[i:i + size], ordered=False)

# ---------- categories (one document per category, subcategories embedded) ----------
cats = load("categories.json")
cat_docs, cat_info = [], {}
for c in cats:
    margins = [s["profit_margin"] for s in c["subcategories"]]
    avg = round(sum(margins) / len(margins), 4)
    cat_docs.append({"_id": c["category_id"], "name": c["name"],
                     "subcategories": c["subcategories"], "avg_profit_margin": avg})
    cat_info[c["category_id"]] = {"category_id": c["category_id"], "name": c["name"],
                                  "avg_profit_margin": avg}
insert_batches(db.categories, cat_docs)

# ---------- products (category summary and price history embedded) ----------
prods = load("products.json")
prod_docs, prod_cat = [], {}
for p in prods:
    prod_docs.append({
        "_id": p["product_id"], "name": p["name"],
        "category": cat_info[p["category_id"]],
        "price": p["base_price"], "stock": p["current_stock"],
        "is_active": p["is_active"], "created": dt(p["creation_date"]),
        "price_history": [{"price": h["price"], "date": dt(h["date"])} for h in p["price_history"]],
    })
    prod_cat[p["product_id"]] = p["category_id"]
insert_batches(db.products, prod_docs)

# ---------- users ----------
users = load("users.json")
user_docs = [{"_id": u["user_id"], "geo": u["geo_data"],
              "registered": dt(u["registration_date"]),
              "registration_month": u["registration_date"][:7],
              "last_active": dt(u["last_active"])} for u in users]
insert_batches(db.users, user_docs)

# ---------- transactions (items embedded, category_id copied into each item) ----------
txns = load("transactions.json")
txn_docs = []
for t in txns:
    items = [{"product_id": i["product_id"], "category_id": prod_cat[i["product_id"]],
              "quantity": i["quantity"], "unit_price": i["unit_price"],
              "subtotal": i["subtotal"]} for i in t["items"]]
    txn_docs.append({"_id": t["transaction_id"], "user_id": t["user_id"],
                     "session_id": t["session_id"], "timestamp": dt(t["timestamp"]),
                     "items": items, "subtotal": t["subtotal"], "discount": t["discount"],
                     "total": t["total"], "payment_method": t["payment_method"],
                     "status": t["status"]})
insert_batches(db.transactions, txn_docs)

# ---------- indexes ----------
db.transactions.create_index([("user_id", 1), ("timestamp", -1)])
db.transactions.create_index("timestamp")
db.transactions.create_index("items.product_id")
db.transactions.create_index("items.category_id")
db.products.create_index("category.category_id")
db.users.create_index("registration_month")
db.users.create_index("geo.country")

for name in ["categories", "products", "users", "transactions"]:
    print(f"{name:13s} {db[name].count_documents({}):>8,}")

Writing /home/jovyan/work/load_mongo.py


In [5]:
!python /home/jovyan/work/load_mongo.py

import pymongo
db = pymongo.MongoClient("mongodb://mongo:27017")["ecommerce"]
st = db.command("dbstats", scale=1024*1024)
print({k: round(st[k], 1) for k in ["dataSize", "storageSize", "indexSize"]}, "MB")

categories          25
products         5,000
users           10,000
transactions   100,000
{'dataSize': 47.0, 'storageSize': 8.5, 'indexSize': 8.8} MB


In [6]:
%%writefile /home/jovyan/work/mongo_pipelines.py
import sys, time
import pymongo
import pandas as pd

URI = sys.argv[1] if len(sys.argv) > 1 else "mongodb://mongo:27017"
db = pymongo.MongoClient(URI)["ecommerce"]

# Pipeline 1 - Revenue analytics: gross revenue, units and estimated profit per category, with monthly trend.
# items are embedded and carry category_id, so no $lookup is needed to reach the category.
P1_REVENUE_BY_CATEGORY = [
    {"$unwind": "$items"},
    {"$group": {"_id": {"cat": "$items.category_id",
                        "month": {"$dateToString": {"format": "%Y-%m", "date": "$timestamp"}}},
                "revenue": {"$sum": "$items.subtotal"},
                "units": {"$sum": "$items.quantity"}}},
    {"$sort": {"_id.month": 1}},
    {"$group": {"_id": "$_id.cat",
                "total_revenue": {"$sum": "$revenue"},
                "units": {"$sum": "$units"},
                "monthly": {"$push": {"month": "$_id.month", "revenue": {"$round": ["$revenue", 2]}}}}},
    {"$lookup": {"from": "categories", "localField": "_id", "foreignField": "_id", "as": "cat"}},
    {"$unwind": "$cat"},
    {"$project": {"_id": 0, "category_id": "$_id", "category": "$cat.name",
                  "total_revenue": {"$round": ["$total_revenue", 2]}, "units": 1,
                  "est_profit": {"$round": [{"$multiply": ["$total_revenue", "$cat.avg_profit_margin"]}, 2]},
                  "monthly": 1}},
    {"$sort": {"total_revenue": -1}},
    {"$limit": 10},
]

# Pipeline 2 - Basic user segmentation: per-user orders and spend, then 4 equal-sized spend segments.
P2_USER_SEGMENTS = [
    {"$group": {"_id": "$user_id", "orders": {"$sum": 1}, "spend": {"$sum": "$total"}}},
    {"$bucketAuto": {"groupBy": "$spend", "buckets": 4,
                     "output": {"users": {"$sum": 1}, "avg_orders": {"$avg": "$orders"},
                                "avg_spend": {"$avg": "$spend"}, "total_spend": {"$sum": "$spend"}}}},
]

if __name__ == "__main__":
    pd.set_option("display.width", 220)
    pd.set_option("display.max_columns", 20)

    t = time.time()
    r1 = list(db.transactions.aggregate(P1_REVENUE_BY_CATEGORY))
    t1 = time.time() - t
    base = pd.DataFrame(r1).drop(columns="monthly")
    months = pd.DataFrame([{m["month"]: m["revenue"] for m in r["monthly"]} for r in r1])
    print("=== Pipeline 1: top 10 categories by gross revenue ===")
    print(pd.concat([base, months], axis=1).to_string(index=False))
    print(f"(ran in {t1:.2f}s)\n")

    t = time.time()
    r2 = list(db.transactions.aggregate(P2_USER_SEGMENTS))
    t2 = time.time() - t
    seg = pd.DataFrame([{"segment": f"S{i+1}", "spend_from": r["_id"]["min"], "spend_to": r["_id"]["max"],
                         "users": r["users"], "avg_orders": round(r["avg_orders"], 2),
                         "avg_spend": round(r["avg_spend"], 2), "total_spend": round(r["total_spend"], 2)}
                        for i, r in enumerate(r2)])
    seg["share_of_revenue_%"] = (100 * seg["total_spend"] / seg["total_spend"].sum()).round(1)
    print("=== Pipeline 2: user segments by total spend (S1 lowest, S4 highest) ===")
    print(seg.to_string(index=False))
    print(f"(ran in {t2:.2f}s)")

Writing /home/jovyan/work/mongo_pipelines.py


In [7]:
!python /home/jovyan/work/mongo_pipelines.py

=== Pipeline 1: top 10 categories by gross revenue ===
 units category_id                   category  total_revenue  est_profit    2026-08    2026-09    2026-10   2026-11
 21244     cat_004             Kelly-Santiago     5452248.89  1123163.27 1513820.61 1827264.20 1911387.54 199776.54
 19294     cat_005            Cameron-Parsons     5123600.74  1408990.20 1433966.54 1712066.84 1771369.00 206198.36
 19993     cat_014 Anderson, Walls and Duncan     5087437.24  1246422.12 1382618.76 1752162.16 1764224.04 188432.28
 18295     cat_024 Patterson, Smith and Jones     4818522.00  1301000.94 1317943.20 1610225.88 1707992.09 182360.83
 18286     cat_021               Mitchell-Kim     4801906.00  1044414.56 1334673.74 1625762.27 1648137.36 193332.63
 19249     cat_018               Phillips Inc     4790386.85  1188015.94 1340310.50 1623361.49 1611746.07 214968.79
 20151     cat_017                Hancock Inc     4771081.84  1383613.73 1353238.24 1599100.50 1635503.21 183239.89
 19349     cat_01